## MD3 Arnis Priedītis

Trenēšanas kods, ir izmainīti daži hiperparametri. Ar savienojumu uz Google Colab T4 GPU trenēšana aizņēma ~4-5h, paspēja 8000 iterācijām tikt cauri.

In [ ]:
import torch
import torch.nn as nn
from torch.nn import functional as F
import math
import matplotlib.pyplot as plt

# hyperparameters
batch_size = 512 # 16 # how many independent sequences will we process in parallel?
block_size = 32 # what is the maximum context length for predictions?
max_iters = 30000
eval_interval = 100
learning_rate = 1e-3
device = 'cuda' if torch.cuda.is_available() else 'cpu'
eval_iters = 200
n_embd = 256 # 64
n_head = 8 # 4
n_layer = 16
dropout = 0.2
# ------------

torch.manual_seed(1337)

#wget https://raw.githubusercontent.com/karpathy/char-rnn/master/data/tinyshakespeare/input.txt
!wget https://github.com/guntisx/DeepLearningCourse/raw/refs/heads/main/MD3/dataset_lv_small.txt
#with open('input.txt', 'r', encoding='utf-8') as f:
with open('dataset_lv_small.txt', 'r', encoding='utf-8') as f:
    text = f.read()

# here are all the unique characters that occur in this text
chars = sorted(list(set(text)))
vocab_size = len(chars)
# create a mapping from characters to integers
stoi = { ch:i for i,ch in enumerate(chars) }
itos = { i:ch for i,ch in enumerate(chars) }
encode = lambda s: [stoi[c] for c in s] # encoder: take a string, output a list of integers
decode = lambda l: ''.join([itos[i] for i in l]) # decoder: take a list of integers, output a string

# Train and test splits
data = torch.tensor(encode(text), dtype=torch.long)
n = int(0.9*len(data)) # first 90% will be train, rest val
train_data = data[:n]
val_data = data[n:]

# data loading
def get_batch(split):
    # generate a small batch of data of inputs x and targets y
    data = train_data if split == 'train' else val_data
    ix = torch.randint(len(data) - block_size, (batch_size,))
    x = torch.stack([data[i:i+block_size] for i in ix])
    y = torch.stack([data[i+1:i+block_size+1] for i in ix])
    x, y = x.to(device), y.to(device)
    return x, y

@torch.no_grad()
def estimate_loss():
    out = {}
    model.eval()
    for split in ['train', 'val']:
        losses = torch.zeros(eval_iters)
        for k in range(eval_iters):
            X, Y = get_batch(split)
            logits, loss = model(X, Y)
            losses[k] = loss.item()
        out[split] = losses.mean()
    model.train()
    return out

class Head(nn.Module):
    """ one head of self-attention """

    def __init__(self, head_size):
        super().__init__()
        self.key = nn.Linear(n_embd, head_size, bias=False)
        self.query = nn.Linear(n_embd, head_size, bias=False)
        self.value = nn.Linear(n_embd, head_size, bias=False)
        self.register_buffer('tril', torch.tril(torch.ones(block_size, block_size)))

        self.dropout = nn.Dropout(dropout)

    def forward(self, x):
        B,T,C = x.shape
        k = self.key(x)   # (B,T,C)
        q = self.query(x) # (B,T,C)
        # compute attention scores ("affinities")
        wei = q @ k.transpose(-2,-1) * C**-0.5 # (B, T, C) @ (B, C, T) -> (B, T, T)
        wei = wei.masked_fill(self.tril[:T, :T] == 0, float('-inf')) # (B, T, T)
        wei = F.softmax(wei, dim=-1) # (B, T, T)
        wei = self.dropout(wei)
        # perform the weighted aggregation of the values
        v = self.value(x) # (B,T,C)
        out = wei @ v # (B, T, T) @ (B, T, C) -> (B, T, C)
        return out

class MultiHeadAttention(nn.Module):
    """ multiple heads of self-attention in parallel """

    def __init__(self, num_heads, head_size):
        super().__init__()
        self.heads = nn.ModuleList([Head(head_size) for _ in range(num_heads)])
        self.proj = nn.Linear(n_embd, n_embd)
        self.dropout = nn.Dropout(dropout)

    def forward(self, x):
        out = torch.cat([h(x) for h in self.heads], dim=-1)
        out = self.dropout(self.proj(out))
        return out

class FeedFoward(nn.Module):
    """ a simple linear layer followed by a non-linearity """

    def __init__(self, n_embd):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(n_embd, 4 * n_embd),
            nn.ReLU(),
            nn.Linear(4 * n_embd, n_embd),
            nn.Dropout(dropout),
        )

    def forward(self, x):
        return self.net(x)

class Block(nn.Module):
    """ Transformer block: communication followed by computation """

    def __init__(self, n_embd, n_head):
        # n_embd: embedding dimension, n_head: the number of heads we'd like
        super().__init__()
        head_size = n_embd // n_head
        self.sa = MultiHeadAttention(n_head, head_size)
        self.ffwd = FeedFoward(n_embd)
        self.ln1 = nn.LayerNorm(n_embd)
        self.ln2 = nn.LayerNorm(n_embd)

    def forward(self, x):
        x = x + self.sa(self.ln1(x))
        x = x + self.ffwd(self.ln2(x))
        return x

# super simple bigram model
class BigramLanguageModel(nn.Module):

    def __init__(self):
        super().__init__()
        # each token directly reads off the logits for the next token from a lookup table
        self.token_embedding_table = nn.Embedding(vocab_size, n_embd)
        self.position_embedding_table = nn.Embedding(block_size, n_embd)
        self.blocks = nn.Sequential(*[Block(n_embd, n_head=n_head) for _ in range(n_layer)])
        self.ln_f = nn.LayerNorm(n_embd) # final layer norm
        self.lm_head = nn.Linear(n_embd, vocab_size)

    def forward(self, idx, targets=None):
        B, T = idx.shape

        # idx and targets are both (B,T) tensor of integers
        tok_emb = self.token_embedding_table(idx) # (B,T,C)
        pos_emb = self.position_embedding_table(torch.arange(T, device=device)) # (T,C)
        x = tok_emb + pos_emb # (B,T,C)
        x = self.blocks(x) # (B,T,C)
        x = self.ln_f(x) # (B,T,C)
        logits = self.lm_head(x) # (B,T,vocab_size)

        if targets is None:
            loss = None
        else:
            B, T, C = logits.shape
            logits = logits.view(B*T, C)
            targets = targets.view(B*T)
            loss = F.cross_entropy(logits, targets)

        return logits, loss

    def generate(self, idx, max_new_tokens):
        # idx is (B, T) array of indices in the current context
        for _ in range(max_new_tokens):
            # crop idx to the last block_size tokens
            idx_cond = idx[:, -block_size:]
            # get the predictions
            logits, loss = self(idx_cond)
            # focus only on the last time step
            logits = logits[:, -1, :] # becomes (B, C)
            # apply softmax to get probabilities
            probs = F.softmax(logits, dim=-1) # (B, C)
            # sample from the distribution
            idx_next = torch.multinomial(probs, num_samples=1) # (B, 1)
            # append sampled index to the running sequence
            idx = torch.cat((idx, idx_next), dim=1) # (B, T+1)
        return idx

model = BigramLanguageModel()
m = model.to(device)
# print the number of parameters in the model
print(sum(p.numel() for p in m.parameters())/1e6, 'M parameters')

loss_values = []  # List to store loss values

# create a PyTorch optimizer
optimizer = torch.optim.AdamW(model.parameters(), lr=learning_rate)

for iter in range(max_iters):

    # every once in a while evaluate the loss on train and val sets
    if iter % eval_interval == 0 or iter == max_iters - 1:
        losses = estimate_loss()
        #print(f"step {iter}: train loss {losses['train']:.4f}, val loss {losses['val']:.4f}")
        print(f"step {iter}: train loss {losses['train']:.4f}, val loss {losses['val']:.4f} (perplexity {1/(math.exp(-1*losses['val'])):.2f} / {vocab_size} = {(1/(math.exp(-1*losses['val'])))/(vocab_size/100):.2f}% uncertainty)")
        loss_values.append([losses['train'],losses['val']])  # Store averaged loss value
        torch.save(model.state_dict(), f"checkpoint_{iter}.pt")
    # sample a batch of data
    xb, yb = get_batch('train')

    # evaluate the loss
    logits, loss = model(xb, yb)
    optimizer.zero_grad(set_to_none=True)
    loss.backward()
    optimizer.step()

# Plot the training loss
plt.plot(loss_values, label='Training Loss')
plt.show()

# generate from the model
context = torch.zeros((1, 1), dtype=torch.long, device=device)
print(decode(m.generate(context, max_new_tokens=2000)[0].tolist()))

# Learning training data
#ROMEO:
#Is she a Capulet?
#O dear account! my life is my foe's debt.
context = torch.tensor(encode("Uz sliekšņa stāvēja"), dtype=torch.long).unsqueeze(0).to(device)
print("--- ", decode(m.generate(context, max_new_tokens=100)[0].tolist()))
print("--- ", decode(m.generate(context, max_new_tokens=100)[0].tolist()))
print("--- ", decode(m.generate(context, max_new_tokens=100)[0].tolist()))

model.eval() # This disables dropout

context = torch.tensor(encode("Uz sliekšņa stāvēja"), dtype=torch.long).unsqueeze(0).to(device)
print("*** ", decode(m.generate(context, max_new_tokens=100)[0].tolist()))
print("*** ", decode(m.generate(context, max_new_tokens=100)[0].tolist()))
print("*** ", decode(m.generate(context, max_new_tokens=100)[0].tolist()))

def generateTop(model, idx, max_new_tokens):
        # idx is (B, T) array of indices in the current context
        for _ in range(max_new_tokens):
            # crop idx to the last block_size tokens
            idx_cond = idx[:, -block_size:]
            # get the predictions
            logits, loss = model.forward(idx_cond)
            # focus only on the last time step
            logits = logits[:, -1, :] # becomes (B, C)
            # apply softmax to get probabilities
            probs = F.softmax(logits, dim=-1) # (B, C)
            # sample from the distribution
            #idx_next = torch.multinomial(probs, num_samples=1) # (B, 1)
            _, idx_next = torch.topk(probs, k=1, dim=-1)
            # append sampled index to the running sequence
            idx = torch.cat((idx, idx_next), dim=1) # (B, T+1)
        return idx

print("+++ ", decode(generateTop(model,context, max_new_tokens=100)[0].tolist()))
print("+++ ", decode(generateTop(model,context, max_new_tokens=100)[0].tolist()))
print("+++ ", decode(generateTop(model,context, max_new_tokens=100)[0].tolist()))

#model.train()

--2026-05-30 10:25:16--  https://github.com/guntisx/DeepLearningCourse/raw/refs/heads/main/MD3/dataset_lv_small.txt
Resolving github.com (github.com)... 20.205.243.166
Connecting to github.com (github.com)|20.205.243.166|:443... connected.
HTTP request sent, awaiting response... 302 Found
Location: https://raw.githubusercontent.com/guntisx/DeepLearningCourse/refs/heads/main/MD3/dataset_lv_small.txt [following]
--2026-05-30 10:25:16--  https://raw.githubusercontent.com/guntisx/DeepLearningCourse/refs/heads/main/MD3/dataset_lv_small.txt
Resolving raw.githubusercontent.com (raw.githubusercontent.com)... 185.199.108.133, 185.199.109.133, 185.199.110.133, ...
Connecting to raw.githubusercontent.com (raw.githubusercontent.com)|185.199.108.133|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 415832 (406K) [text/plain]
Saving to: ‘dataset_lv_small.txt’

dataset_lv_small.tx 100%[===================>] 406.09K  --.-KB/s    in 0.003s  

2026-05-30 10:25:16 (141 MB/s) - ‘da

KeyboardInterrupt: 

### Pārbaudam checkpoint 8000 uz frāzi "Uz sliekšņa stāvēja"

### MD3: Here we generate text from any prompt

It is not yet a ChatGPT, because it is not yet fine-tuned on the Instruction dataset containing Questions and Answer pairs

Velreiz ieliku importus un klašu definīcijas, lai varētu vēl nākamo koda bloku palaist bez pārtrenēšanas.

In [ ]:
import torch
import torch.nn as nn
from torch.nn import functional as F
import math
import matplotlib.pyplot as plt

# hyperparameters
batch_size = 512 # 16 # how many independent sequences will we process in parallel?
block_size = 32 # what is the maximum context length for predictions?
max_iters = 30000
eval_interval = 100
learning_rate = 1e-3
device = 'cuda' if torch.cuda.is_available() else 'cpu'
eval_iters = 200
n_embd = 256 # 64
n_head = 8 # 4
n_layer = 16
dropout = 0.2
# ------------

torch.manual_seed(1337)

#wget https://raw.githubusercontent.com/karpathy/char-rnn/master/data/tinyshakespeare/input.txt
!wget https://github.com/guntisx/DeepLearningCourse/raw/refs/heads/main/MD3/dataset_lv_small.txt
#with open('input.txt', 'r', encoding='utf-8') as f:
with open('dataset_lv_small.txt', 'r', encoding='utf-8') as f:
    text = f.read()

# here are all the unique characters that occur in this text
chars = sorted(list(set(text)))
vocab_size = len(chars)
# create a mapping from characters to integers
stoi = { ch:i for i,ch in enumerate(chars) }
itos = { i:ch for i,ch in enumerate(chars) }
encode = lambda s: [stoi[c] for c in s] # encoder: take a string, output a list of integers
decode = lambda l: ''.join([itos[i] for i in l]) # decoder: take a list of integers, output a string

# Train and test splits
data = torch.tensor(encode(text), dtype=torch.long)
n = int(0.9*len(data)) # first 90% will be train, rest val
train_data = data[:n]
val_data = data[n:]

# data loading
def get_batch(split):
    # generate a small batch of data of inputs x and targets y
    data = train_data if split == 'train' else val_data
    ix = torch.randint(len(data) - block_size, (batch_size,))
    x = torch.stack([data[i:i+block_size] for i in ix])
    y = torch.stack([data[i+1:i+block_size+1] for i in ix])
    x, y = x.to(device), y.to(device)
    return x, y

@torch.no_grad()
def estimate_loss():
    out = {}
    model.eval()
    for split in ['train', 'val']:
        losses = torch.zeros(eval_iters)
        for k in range(eval_iters):
            X, Y = get_batch(split)
            logits, loss = model(X, Y)
            losses[k] = loss.item()
        out[split] = losses.mean()
    model.train()
    return out

class Head(nn.Module):
    """ one head of self-attention """

    def __init__(self, head_size):
        super().__init__()
        self.key = nn.Linear(n_embd, head_size, bias=False)
        self.query = nn.Linear(n_embd, head_size, bias=False)
        self.value = nn.Linear(n_embd, head_size, bias=False)
        self.register_buffer('tril', torch.tril(torch.ones(block_size, block_size)))

        self.dropout = nn.Dropout(dropout)

    def forward(self, x):
        B,T,C = x.shape
        k = self.key(x)   # (B,T,C)
        q = self.query(x) # (B,T,C)
        # compute attention scores ("affinities")
        wei = q @ k.transpose(-2,-1) * C**-0.5 # (B, T, C) @ (B, C, T) -> (B, T, T)
        wei = wei.masked_fill(self.tril[:T, :T] == 0, float('-inf')) # (B, T, T)
        wei = F.softmax(wei, dim=-1) # (B, T, T)
        wei = self.dropout(wei)
        # perform the weighted aggregation of the values
        v = self.value(x) # (B,T,C)
        out = wei @ v # (B, T, T) @ (B, T, C) -> (B, T, C)
        return out

class MultiHeadAttention(nn.Module):
    """ multiple heads of self-attention in parallel """

    def __init__(self, num_heads, head_size):
        super().__init__()
        self.heads = nn.ModuleList([Head(head_size) for _ in range(num_heads)])
        self.proj = nn.Linear(n_embd, n_embd)
        self.dropout = nn.Dropout(dropout)

    def forward(self, x):
        out = torch.cat([h(x) for h in self.heads], dim=-1)
        out = self.dropout(self.proj(out))
        return out

class FeedFoward(nn.Module):
    """ a simple linear layer followed by a non-linearity """

    def __init__(self, n_embd):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(n_embd, 4 * n_embd),
            nn.ReLU(),
            nn.Linear(4 * n_embd, n_embd),
            nn.Dropout(dropout),
        )

    def forward(self, x):
        return self.net(x)

class Block(nn.Module):
    """ Transformer block: communication followed by computation """

    def __init__(self, n_embd, n_head):
        # n_embd: embedding dimension, n_head: the number of heads we'd like
        super().__init__()
        head_size = n_embd // n_head
        self.sa = MultiHeadAttention(n_head, head_size)
        self.ffwd = FeedFoward(n_embd)
        self.ln1 = nn.LayerNorm(n_embd)
        self.ln2 = nn.LayerNorm(n_embd)

    def forward(self, x):
        x = x + self.sa(self.ln1(x))
        x = x + self.ffwd(self.ln2(x))
        return x

# super simple bigram model
class BigramLanguageModel(nn.Module):

    def __init__(self):
        super().__init__()
        # each token directly reads off the logits for the next token from a lookup table
        self.token_embedding_table = nn.Embedding(vocab_size, n_embd)
        self.position_embedding_table = nn.Embedding(block_size, n_embd)
        self.blocks = nn.Sequential(*[Block(n_embd, n_head=n_head) for _ in range(n_layer)])
        self.ln_f = nn.LayerNorm(n_embd) # final layer norm
        self.lm_head = nn.Linear(n_embd, vocab_size)

    def forward(self, idx, targets=None):
        B, T = idx.shape

        # idx and targets are both (B,T) tensor of integers
        tok_emb = self.token_embedding_table(idx) # (B,T,C)
        pos_emb = self.position_embedding_table(torch.arange(T, device=device)) # (T,C)
        x = tok_emb + pos_emb # (B,T,C)
        x = self.blocks(x) # (B,T,C)
        x = self.ln_f(x) # (B,T,C)
        logits = self.lm_head(x) # (B,T,vocab_size)

        if targets is None:
            loss = None
        else:
            B, T, C = logits.shape
            logits = logits.view(B*T, C)
            targets = targets.view(B*T)
            loss = F.cross_entropy(logits, targets)

        return logits, loss

    def generate(self, idx, max_new_tokens):
        # idx is (B, T) array of indices in the current context
        for _ in range(max_new_tokens):
            # crop idx to the last block_size tokens
            idx_cond = idx[:, -block_size:]
            # get the predictions
            logits, loss = self(idx_cond)
            # focus only on the last time step
            logits = logits[:, -1, :] # becomes (B, C)
            # apply softmax to get probabilities
            probs = F.softmax(logits, dim=-1) # (B, C)
            # sample from the distribution
            idx_next = torch.multinomial(probs, num_samples=1) # (B, 1)
            # append sampled index to the running sequence
            idx = torch.cat((idx, idx_next), dim=1) # (B, T+1)
        return idx

model = BigramLanguageModel()

--2026-05-31 17:21:48--  https://github.com/guntisx/DeepLearningCourse/raw/refs/heads/main/MD3/dataset_lv_small.txt
Resolving github.com (github.com)... 140.82.114.4
Connecting to github.com (github.com)|140.82.114.4|:443... connected.
HTTP request sent, awaiting response... 302 Found
Location: https://raw.githubusercontent.com/guntisx/DeepLearningCourse/refs/heads/main/MD3/dataset_lv_small.txt [following]
--2026-05-31 17:21:48--  https://raw.githubusercontent.com/guntisx/DeepLearningCourse/refs/heads/main/MD3/dataset_lv_small.txt
Resolving raw.githubusercontent.com (raw.githubusercontent.com)... 185.199.108.133, 185.199.109.133, 185.199.110.133, ...
Connecting to raw.githubusercontent.com (raw.githubusercontent.com)|185.199.108.133|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 415832 (406K) [text/plain]
Saving to: ‘dataset_lv_small.txt’

dataset_lv_small.tx 100%[===================>] 406.09K  --.-KB/s    in 0.04s   

2026-05-31 17:21:48 (10.7 MB/s) - ‘datas

In [ ]:
!wget https://github.com/arnis-prieditis/dzila_masinmacisanas_MD3_faili/blob/main/checkpoint_8000.pt?raw=true -O checkpoint_8000.pt

--2026-05-31 18:30:44--  https://github.com/arnis-prieditis/dzila_masinmacisanas_MD3_faili/blob/main/checkpoint_8000.pt?raw=true
Resolving github.com (github.com)... 140.82.114.3
Connecting to github.com (github.com)|140.82.114.3|:443... connected.
HTTP request sent, awaiting response... 302 Found
Location: https://github.com/arnis-prieditis/dzila_masinmacisanas_MD3_faili/raw/refs/heads/main/checkpoint_8000.pt [following]
--2026-05-31 18:30:44--  https://github.com/arnis-prieditis/dzila_masinmacisanas_MD3_faili/raw/refs/heads/main/checkpoint_8000.pt
Reusing existing connection to github.com:443.
HTTP request sent, awaiting response... 302 Found
Location: https://raw.githubusercontent.com/arnis-prieditis/dzila_masinmacisanas_MD3_faili/refs/heads/main/checkpoint_8000.pt [following]
--2026-05-31 18:30:44--  https://raw.githubusercontent.com/arnis-prieditis/dzila_masinmacisanas_MD3_faili/refs/heads/main/checkpoint_8000.pt
Resolving raw.githubusercontent.com (raw.githubusercontent.com)... 1

In [ ]:
model.load_state_dict(torch.load( f"checkpoint_8000.pt", map_location=torch.device('cpu')))
model.eval()

def generate_with_prompt(prompt, max_new_tokens):
    prompt_encoded = torch.tensor(encode(prompt), dtype=torch.long).unsqueeze(0).to(device)
    generated = model.generate(prompt_encoded, max_new_tokens=max_new_tokens)
    generated_text = decode(generated.squeeze().tolist())
    return generated_text

while True:
    prompt = input("Ievadi promtu vai q, lai pabeigtu sarunu: ")
    if prompt.lower() == 'q':
        break

    generated_text = generate_with_prompt(prompt, max_new_tokens=50)
    print(generated_text)
    print("-----------")


Ievadi promtu vai q, lai pabeigtu sarunu: Uz sliekšņa stāvēja
Uz sliekšņa stāvēja Ketrina Bārklija.
Viņa ienāca istabā un tuvojās g
-----------
Ievadi promtu vai q, lai pabeigtu sarunu: Uz sliekšņa stāvēja
Uz sliekšņa stāvēja Ketrina Bārklija.
Izdzirdu soļus vestibilā.
Es pi
-----------
Ievadi promtu vai q, lai pabeigtu sarunu: Uz sliekšņa stāvēja
Uz sliekšņa stāvēja Ketrina Bārklija.
Viņa ienāca istabā un tuvojās g
-----------
Ievadi promtu vai q, lai pabeigtu sarunu: Uz sliekšņa stāvēja
Uz sliekšņa stāvēja Ketrina Bārklija.
Viņa ienāca istabā un tuvojās g
-----------
Ievadi promtu vai q, lai pabeigtu sarunu: Uz sliekšņa stāvēja
Uz sliekšņa stāvēja Ketrina, un es izkāpu no gultas un pa auksto grīd
-----------
Ievadi promtu vai q, lai pabeigtu sarunu: Es gribu
Es gribu tevi.
Esmu kā neprātīgs pēc tevis."
"Tu patiešām 
-----------
Ievadi promtu vai q, lai pabeigtu sarunu: Viņa
Viņa sacīja.
"Jūs tīri labi iztiksiet bez manis."
"Es 
-----------
Ievadi promtu vai q, lai pabeigtu sarunu: q
